# Direct MNA / SSN component benchmark

This notebook records small-grid comparisons between each component's original subcomponent implementation and its optional direct/optimized constructor. Legacy constructors remain the default. The broad EMT fixture uses 0.5 s; most use Δt = 50 μs (10,000 steps), and the DC π-line uses Δt = 10 μs (50,000 steps). GFL uses the dedicated controlled 10 s / 100 μs / 100,000-step load-breaker case. The DP shunt uses 0.04 s at Δt = 10 μs (4,000 steps). Timings are arithmetic means of five runs.

The trace error is the maximum absolute difference over logged values; values are rounded by the CSV logger. Scaled error is max absolute error divided by the largest legacy trace magnitude (or stated physical bases where needed). The DP transient-stability generator timing is the mean of five runs. This is a component-level microbenchmark, not a prediction of large-grid speedup.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = [
    ("EMT Ph3 RxLine", 0.305558091, 0.223337342, 0.0, 0.0, "5-run mean"),
    ("EMT Ph3 NetworkInjection", 0.154227325, 0.129811750, 0.0, 0.0, "5-run mean"),
    ("EMT Ph3 Shunt", 0.243379642, 0.234429067, 0.0, 0.0, "5-run mean"),
    ("EMT Ph3 PiLine", 0.418451892, 0.276413533, 1.00001e-6, 7.41167e-12, "5-run mean"),
    ("EMT Ph3 RXLoad", 0.234877159, 0.178949867, 0.0, 0.0, "5-run mean"),
    ("EMT Ph3 Transformer", 0.349559892, 0.322250417, 1.29e-4, 1.29e-9, "5-run mean"),
    (
        "EMT Ph3 HalfDecouplingLine",
        1.094122517,
        1.173413058,
        0.0,
        0.0,
        "5-run mean; slower direct",
    ),
    ("EMT DC PiLine", 1.920753375, 0.860760600, 1.0e-6, 4.48061e-11, "5-run mean"),
    ("EMT Ph1 PiLine", 0.115979000, 0.084108358, 1.0e-6, 8.03574e-12, "5-run mean"),
    ("EMT Ph1 VoltageSourceRamp", 0.068661908, 0.063428367, 0.0, 0.0, "5-run mean"),
    (
        "EMT Ph3 SynchronGeneratorIdeal",
        0.137654117,
        0.119820492,
        1.00001e-6,
        1.22475e-11,
        "5-run mean",
    ),
    (
        "EMT Ph3 SynchronGeneratorTrStab",
        0.471021867,
        0.443954650,
        0.0,
        0.0,
        "5-run mean",
    ),
    (
        "EMT Ph3 GFL (no transformer)",
        10.8008622,
        9.8353444,
        0.0,
        0.0,
        "5-run mean; controlled 10 s; all traces exact",
    ),
    (
        "EMT Ph3 AvVoltageSourceInverterDQ (open-loop)",
        0.943857,
        0.776564,
        1.00001e-6,
        1.00338e-11,
        "5 runs",
    ),
    (
        "EMT Ph3 VSIVoltageControlVCO (open-loop)",
        0.704427,
        0.536352,
        1.00001e-6,
        1.00338e-11,
        "5 runs",
    ),
    (
        "DP Ph1 NetworkInjection",
        0.0208228,
        0.0143389,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    (
        "DP Ph1 RXLoad (R + L)",
        0.0220163,
        0.0173861,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    (
        "DP Ph1 RXLoadSwitch",
        0.0493953,
        0.0471048,
        3.6e-6,
        9.01388e-9,
        "5-run mean; full-trace relative errors V=9.0e-9, I=0",
    ),
    (
        "DP Ph1 Shunt (G + C)",
        0.0217044,
        0.0174893,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    ("DP Ph1 PiLine", 0.0384890, 0.0266584, 0.0, 0.0, "5-run mean; final validation"),
    ("DP Ph1 RxLine", 0.0294843, 0.0188155, 0.0, 0.0, "5-run mean; final validation"),
    (
        "DP Ph1 VoltageSourceRamp",
        0.00446122,
        0.00442303,
        0.0,
        0.0,
        "5-run mean; neutral",
    ),
    (
        "DP Ph1 SynchronGeneratorIdeal",
        0.0192293,
        0.00779498,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    (
        "DP Ph1 SynchronGeneratorTrStab",
        0.0257983,
        0.0143631,
        1.09e-6,
        7.7576e-6,
        "5-run mean; base-scaled trace error",
    ),
    ("DP Ph1 PQLoadCS", 0.0127989, 0.0140711, 0.0, 0.0, "5-run mean; direct is slower"),
    (
        "DP Ph1 SVC (0.04 s, pre-control)",
        0.0316247,
        0.000754308,
        0.0,
        0.0,
        "5-run mean; before control acts",
    ),
    (
        "DP Ph3 NetworkInjection",
        0.0241308,
        0.0189429,
        0.0,
        0.0,
        "5-run mean; final validation",
    ),
    ("DP Ph3 PiLine", 0.262044, 0.103471, 0.0, 0.0, "5-run mean; final validation"),
    (
        "DP Ph1 Transformer (losses enabled)",
        0.0536820,
        0.0394841,
        0.000873139,
        0.000334667,
        "5-run mean; base-scaled errors; noisy legacy",
    ),
    (
        "DP Ph3 Transformer",
        0.279729,
        0.118062,
        0.000502309,
        0.000352833,
        "5-run mean; base-scaled trace errors",
    ),
    (
        "DP Ph1 AvVoltageSourceInverterDQ (no transformer)",
        0.289608,
        0.265096,
        0.0,
        0.0,
        "5-run mean; open-loop and control-on 5 ms traces exact",
    ),
    (
        "DP Ph1 AvVoltageSourceInverterDQ (with transformer)",
        0.0317227,
        0.0311276,
        0.823868,
        0.000613281,
        "5-run mean; 0.04 s; max relative current error 0.0613%",
    ),
]
df = pd.DataFrame(
    rows,
    columns=[
        "Component",
        "Before fix / legacy mean (s)",
        "After fix / direct mean (s)",
        "Max abs trace error",
        "Scaled trace error",
        "Timing sample",
    ],
)
df["Speedup (%)"] = (
    100
    * (df["Before fix / legacy mean (s)"] - df["After fix / direct mean (s)"])
    / df["Before fix / legacy mean (s)"]
)
display(
    df.style.format(
        {
            "Before fix / legacy mean (s)": "{:.6f}",
            "After fix / direct mean (s)": "{:.6f}",
            "Max abs trace error": "{:.3g}",
            "Scaled trace error": "{:.3g}",
            "Speedup (%)": "{:+.2f}%",
        }
    )
)

In [ ]:
plot_df = df[df["Timing sample"].str.startswith("5")].set_index("Component")
ax = plot_df[["Before fix / legacy mean (s)", "After fix / direct mean (s)"]].plot.bar(
    figsize=(13, 5), color=["#6b7280", "#2563eb"]
)
ax.set_ylabel("Wall time (s)")
ax.set_title("Legacy vs direct component implementation — five-run means")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

## Interpretation and coverage

Direct mode improves most tested passive EMT cases. In the five-run means the half-decoupling line is slower in direct mode, so that option should not be selected for this component. GFL direct mode now matches all 100,001 logged samples exactly in each of five controlled 10 s comparisons and is about 8.9% faster. Do not infer grid-scale performance from component microbenchmarks.

The table labels the legacy implementation as before-fix and the direct implementation as after-fix. DP means are from the five-run rerun at 0.04 s and 10 μs. Most direct paths are faster, while DP PQLoadCS is slower in this small fixture. SVC timing covers only the pre-control interval and is not a claim about dynamic-controller performance or equivalence. EMT component rows were refreshed from five runs of the broad comparison executable; GFL uses five full-length controlled runs per mode.

## Remaining DP composite coverage

| Component | Direct/no-electrical-child path | Validation status |
|---|---|---|
| DP Ph1 AvVoltageSourceInverterDQ | Implemented with or without transformer | No-transformer open-loop and controlled 5 ms traces match; with transformer, voltage trace is exact and max relative current error is 0.0613%; five-run timing is included in the table |
| DP Ph1 RXLoadSwitch | Implemented | 0.04 s full trace comparison passes (relative error 9.0e-9 voltage, zero current); five-run mean is 47.5 ms legacy vs 42.0 ms direct |
| DP Ph1 SVC | Present in working tree | Pre-control 0.04 s traces match; beyond 0.1 s the legacy model itself becomes unstable for the tested offset, so dynamic-control parity is not claimed |

For transformer-enabled AvVoltageSourceInverterDQ, the direct parent keeps filter stamps compact while retaining the voltage-source and transformer MNA children. Their per-step RHS contributions use cached affected rows; the five-run timing is a small 0.04 s microbenchmark.